# DragonHPC GPU Multiprocessing Tutorial for LLM Inference

**Estimated time:** ~35-50 minutes  
**Format:** 5 exercises with demonstration code, a short student task, and hidden solutions.

---

## Session goals

This notebook introduces Python multiprocessing with GPUs and PyTorch in
DragonHPC, motivated by distributed LLM inference patterns.

You will practice how to:

- discover GPU resources and map workers to devices
- launch Dragon native processes for GPU work
- use Dragon `Policy` to set process-to-GPU affinity
- reason about single-node vs multi-node placement
- build a toy distributed inference workflow

The examples use lightweight PyTorch computations so the notebook stays
portable and quick to run.

---

## Setup - run this first

Start Jupyter from a Dragon-enabled environment (for example with
`dragon-jupyter`) and run the next cell.

In [ ]:
import os
import socket
import time
import dragon
import multiprocessing as mp
import dragon.native as dn
from dragon.infrastructure.policy import Policy
import torch

# Use Dragon as multiprocessing backend. Safe for reruns in notebooks.
try:
    mp.set_start_method("dragon")
except RuntimeError:
    pass

print("Dragon + PyTorch ready")
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

`dragon.native.Process` works well in Jupyter and supports explicit Dragon
`Policy` objects. For GPU placement, policy options like `gpu_affinity` can
be combined with host placement controls to manage process locality.

---

## Exercise 1 - Discover GPUs and map worker ranks

**Background:**

Many inference systems assign worker rank `r` to GPU `r % num_gpus`.
This gives a simple round-robin mapping within a node.

Demo pattern:

```python
num_gpus = torch.cuda.device_count()
if num_gpus > 0:
    print({r: r % num_gpus for r in range(8)})
else:
    print("No GPU: use CPU fallback")
```

**Your task:**

1. Write `pick_device(rank, num_gpus)`
2. Return `f"cuda:{rank % num_gpus}"` when `num_gpus > 0`, otherwise `"cpu"`
3. Print mapping for ranks 0 through 11

In [ ]:
# -- Exercise 1 -- your code here -----------------------------------------------

def pick_device(rank, num_gpus):
    pass

num_gpus = torch.cuda.device_count()
# for r in range(12):
#     print(r, pick_device(r, num_gpus))

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def pick_device(rank, num_gpus):
    if num_gpus > 0:
        return f"cuda:{rank % num_gpus}"
    return "cpu"

num_gpus = torch.cuda.device_count()
for r in range(12):
    print(r, pick_device(r, num_gpus))
```

</details>

---

## Exercise 2 - Launch GPU workers with Dragon Native Process

**Background:**

A worker can select a device, run a small PyTorch computation, and return
results through an `mp.Queue`.

Demo pattern:

```python
def one_worker(out_q):
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    x = torch.arange(8, device=device, dtype=torch.float32)
    out_q.put((os.getpid(), device, float(x.sum().item())))

q = mp.Queue()
p = dn.Process(target=one_worker, args=(q,))
p.start()
print(q.get())
p.join()
```

**Your task:**

1. Write `gpu_worker(rank, out_q)`
2. Use `pick_device(rank, num_gpus)` from Exercise 1
3. Compute `torch.arange(1000, device=device).sum()` and return `(rank, device, sum)`
4. Launch 4 workers and print all results sorted by rank

In [ ]:
# -- Exercise 2 -- your code here -----------------------------------------------

def gpu_worker(rank, out_q):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def gpu_worker(rank, out_q):
    num_gpus = torch.cuda.device_count()
    device = pick_device(rank, num_gpus)
    x = torch.arange(1000, dtype=torch.float32, device=device)
    out_q.put((rank, device, float(x.sum().item())))

q = mp.Queue()
procs = [dn.Process(target=gpu_worker, args=(r, q)) for r in range(4)]

for p in procs:
    p.start()

results = [q.get() for _ in procs]
for p in procs:
    p.join()

for item in sorted(results):
    print(item)
```

</details>

---

## Exercise 3 - Control GPU affinity with Dragon Policy

**Background:**

Dragon `Policy` lets you control affinity. For GPUs, use `gpu_affinity`.
For example, `Policy(gpu_affinity=[1])` requests placement targeting GPU 1.

Demo pattern:

```python
def report_affinity(out_q):
    out_q.put((os.getpid(), os.environ.get("CUDA_VISIBLE_DEVICES", "unset")))

q = mp.Queue()
pol = Policy(gpu_affinity=[0])
p = dn.Process(target=report_affinity, args=(q,), policy=pol)
p.start()
print(q.get())
p.join()
```

**Your task:**

1. Create two policies: one for GPU 0 and one for GPU 1
2. Launch two workers with those explicit policies
3. Each worker should report `(rank, pid, CUDA_VISIBLE_DEVICES)`
4. Print both reports

If your system has fewer than 2 GPUs, use CPU fallback and still run both
workers (the environment report may be the same).

In [ ]:
# -- Exercise 3 -- your code here -----------------------------------------------

def affinity_worker(rank, out_q):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def affinity_worker(rank, out_q):
    out_q.put((rank, os.getpid(), os.environ.get("CUDA_VISIBLE_DEVICES", "unset")))

q = mp.Queue()
num_gpus = torch.cuda.device_count()
gpu0 = 0
gpu1 = 1 if num_gpus > 1 else 0

policies = [Policy(gpu_affinity=[gpu0]), Policy(gpu_affinity=[gpu1])]
procs = [
    dn.Process(target=affinity_worker, args=(0, q), policy=policies[0]),
    dn.Process(target=affinity_worker, args=(1, q), policy=policies[1]),
]

for p in procs:
    p.start()

for _ in procs:
    print(q.get())

for p in procs:
    p.join()
```

</details>

---

## Exercise 4 - Multi-node placement: host and GPU policies

**Background:**

On a single-node run, all processes start on the same host. On multi-node
runs, default placement is round-robin across allocated nodes. If you need
precise control, combine host placement and GPU affinity in `Policy`.

Demo pattern:

```python
host = socket.gethostname()
with Policy(placement=Policy.Placement.HOST_NAME, host_name=host):
    p = dn.Process(target=lambda: print("on", socket.gethostname()))
    p.start()
    p.join()
```

**Your task:**

1. Write `host_gpu_worker(rank, out_q)` that reports `(rank, hostname, CUDA_VISIBLE_DEVICES)`
2. Create 4 processes
3. For each process, pass an explicit policy with `gpu_affinity=[rank % max(1, num_gpus)]`
4. Collect and print all reports sorted by rank
5. Print a host histogram (how many workers per host)

In [ ]:
# -- Exercise 4 -- your code here -----------------------------------------------

def host_gpu_worker(rank, out_q):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def host_gpu_worker(rank, out_q):
    out_q.put((
        rank,
        socket.gethostname(),
        os.environ.get("CUDA_VISIBLE_DEVICES", "unset"),
    ))

q = mp.Queue()
num_gpus = torch.cuda.device_count()
procs = []
for rank in range(4):
    gpu_id = rank % max(1, num_gpus)
    pol = Policy(gpu_affinity=[gpu_id])
    procs.append(dn.Process(target=host_gpu_worker, args=(rank, q), policy=pol))

for p in procs:
    p.start()

reports = [q.get() for _ in procs]
for p in procs:
    p.join()

for item in sorted(reports):
    print(item)

host_hist = {}
for _, host, _ in reports:
    host_hist[host] = host_hist.get(host, 0) + 1
print("host histogram:", host_hist)
```

</details>

---

## Exercise 5 - Toy multi-GPU PyTorch inference pipeline

**Background:**

This exercise mimics distributed LLM inference structure: prompts are
partitioned across workers, each worker runs a PyTorch forward path on its
assigned device, and results are aggregated in the parent process.

Demo pattern:

```python
def score_prompt(prompt, device):
    x = torch.tensor([ord(c) % 256 for c in prompt], dtype=torch.float32, device=device)
    return float((x.mean() / 255.0).item())
```

**Your task:**

1. Implement `infer_worker(worker_id, prompts, out_q)`
2. Choose device via `pick_device(worker_id, num_gpus)`
3. For each prompt, compute a score with the toy model below and emit `(worker_id, prompt, score, device)`
4. Launch 4 workers over prompt shards
5. Collect all outputs and print results sorted by prompt

In [ ]:
# -- Exercise 5 -- your code here -----------------------------------------------

def toy_model_score(prompt, device):
    x = torch.tensor([ord(c) % 256 for c in prompt], dtype=torch.float32, device=device)
    w = torch.linspace(0.5, 1.5, steps=len(x), device=device)
    return float(((x * w).mean() / 255.0).item())

def infer_worker(worker_id, prompts, out_q):
    pass

prompts = [
    "Summarize DragonHPC multiprocessing in one sentence.",
    "Why are GPUs useful for LLM inference?",
    "How does policy-based affinity help performance?",
    "What is round-robin process placement?",
    "How would you shard prompts across workers?",
    "Name one risk of poor device placement.",
    "What does CUDA_VISIBLE_DEVICES mean?",
    "How can you debug cross-node placement issues?",
]

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def toy_model_score(prompt, device):
    x = torch.tensor([ord(c) % 256 for c in prompt], dtype=torch.float32, device=device)
    w = torch.linspace(0.5, 1.5, steps=len(x), device=device)
    return float(((x * w).mean() / 255.0).item())

def infer_worker(worker_id, prompts, out_q):
    num_gpus = torch.cuda.device_count()
    device = pick_device(worker_id, num_gpus)
    for prompt in prompts:
        score = toy_model_score(prompt, device)
        out_q.put((worker_id, prompt, score, device))

num_workers = 4
shards = [prompts[i::num_workers] for i in range(num_workers)]
q = mp.Queue()
procs = [dn.Process(target=infer_worker, args=(wid, shards[wid], q)) for wid in range(num_workers)]

for p in procs:
    p.start()

all_results = []
for _ in range(len(prompts)):
    all_results.append(q.get())

for p in procs:
    p.join()

for worker_id, prompt, score, device in sorted(all_results, key=lambda t: t[1]):
    print(f"[{device}] worker={worker_id} score={score:.4f} :: {prompt}")
```

</details>

---

## Summary

You built from basic rank-to-device mapping to policy-controlled placement
and a toy distributed inference pipeline with PyTorch.

| Concept | API |
|---|---|
| Dragon backend | `mp.set_start_method("dragon")` |
| Launch process | `dn.Process(...)` |
| GPU mapping | `rank % num_gpus` |
| Explicit GPU affinity | `Policy(gpu_affinity=[...])` |
| Host placement | `Policy(placement=HOST_NAME, host_name=...)` |
| Result aggregation | `mp.Queue()` |
| PyTorch inference | tensor ops on `cuda:N` or CPU fallback |

### Next steps

- Replace the toy scoring function with your real model forward pass.
- Extend policies for per-node, per-GPU rank layouts.
- Add timing metrics per worker to study scaling and placement effects.